# S8 — ground leaf area index from the reprocessed archive

Predicts HF069's plot measurement from S1's HF365 v2. Earlier forecasting work in this
project predicted MODIS `Lai_500m` from HF365 indices, which is one optical retrieval
predicting another; the target here is on the ground.

Rebuild everything: `python -m s8 run`. This notebook reads what that wrote and does not
recompute it, except where a cell says otherwise.

In [1]:
import json

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from s8 import data as s8data, model as s8model, findings as s8findings

pd.set_option("display.width", 150)

SUMMARY = json.loads((s8model.OUT / "results_summary.json").read_text("utf8"))
pairs = s8data.load_pairs()
ladder = pd.read_csv(s8model.OUT / "cv_results.csv")
seasons = pd.read_csv(s8model.OUT / "season_skill.csv")
oof = pd.read_csv(s8model.OUT / "oof_predictions.csv", parse_dates=["date"])

print(f"{len(pairs):,} pairs | {pairs['plot'].nunique()} plots | "
      f"{pairs['campaign_id'].nunique()} campaigns | "
      f"{pairs['date'].min().date()} -> {pairs['date'].max().date()}")

2,894 pairs | 37 plots | 92 campaigns | 2015-07-13 -> 2025-11-20


## The column named `lai` holds PAI

`hf069.xml` is explicit that file 01 reports plant area index, uncorrected for woody
material. The instrument cannot separate a leaf from a branch, so the leaf-off reading is
the woody component. The archive's own site file carries both series and the gap is
visible directly.

S8's target is `pai - min(pai)` per plot — WP0's `lai_subtractive_fx`. The variants
`lai_printed_*` reproduce the source paper's printed equation including its pathology
(it never reaches zero at leaf-off) and are not used.

In [2]:
site = pd.read_csv(s8findings.DATA / "HF069" / "hf069-02-lai-site.csv",
                   parse_dates=["date"])
leaf_off = site[site.doy < 100]
print(f"leaf-off site PAI  : {leaf_off.all_mean.mean():.2f}")
print(f"leaf-off decid LAI : {leaf_off.decid_mean.mean():.2f}")
print(f"woody component    : {leaf_off.all_mean.mean() - leaf_off.decid_mean.mean():.2f}")

fig = go.Figure()
for col, name in [("all_mean", "PAI, uncorrected"), ("decid_mean", "deciduous LAI")]:
    fig.add_scatter(x=site.doy, y=site[col], mode="markers", name=name,
                    marker=dict(size=4, opacity=0.55))
fig.update_layout(template="plotly_white", height=380,
                  margin=dict(l=60, r=20, t=60, b=40),
                  title="hf069-02 site series, all years stacked on day-of-year",
                  xaxis_title="day of year", yaxis_title="m2 / m2")
fig.show()

leaf-off site PAI  : 1.72
leaf-off decid LAI : 0.08
woody component    : 1.64


## Matching

One row is one HF069 reading matched to the nearest usable HF365 v2 observation of the
same plot. The two calendars are near-disjoint by construction — a satellite orbit
against a field crew's schedule — so exact-date joining discards most of the overlap.

The tolerance sweep below is recomputed live. The shipped default is ±3 days: it costs
about 30 % of the sample against ±8 and buys accuracy, and in the shoulder seasons an
8-day gap is itself disagreement rather than measurement error.

In [3]:
g, h = s8data.load_ground(), s8data.load_satellite()
rows = []
for tol in (1, 3, 5, 8, 16):
    m = s8data.add_features(s8data.match(g, h, tol))
    s = s8model.cross_validate(
        s8model.build(m, *s8model.LADDER[s8model.CHOSEN], "lai", "known_plot"))[1]
    rows.append({"tol_days": tol, "pairs": len(m), "campaigns": m.campaign_id.nunique(),
                 "mean_gap_d": round(m.days_apart.mean(), 2),
                 "r2": s["r2"], "rmse": s["rmse"]})
pd.DataFrame(rows)

,tol_days,pairs,campaigns,mean_gap_d,r2,rmse
0,1,1346,45,0.57,0.8898,0.4035
1,3,2894,92,1.55,0.8977,0.4111
2,5,3593,115,2.07,0.8715,0.4517
3,8,3986,129,2.55,0.8772,0.4471
4,16,4304,137,3.18,0.8773,0.4522


## Variance budget, before any model

Where the target's variability actually sits. This is arithmetic on the pair table, not a
fit, and it sets what any model can possibly reach.

In [4]:
for target in ("pai", "lai"):
    vb = s8model.variance_budget(pairs, target)
    print(f"{target:>4}  total {vb['total']:.3f}  | between-date "
          f"{100 * vb['between_date_frac']:5.1f} %  | within-date "
          f"{100 * vb['within_date_frac']:5.1f} %  | static plot offset "
          f"{100 * vb['static_plot_frac']:5.1f} %")

 pai  total 1.797  | between-date  79.2 %  | within-date  20.8 %  | static plot offset  13.2 %
 lai  total 1.653  | between-date  86.5 %  | within-date  13.5 %  | static plot offset   5.6 %


The static plot offset drops from PAI to LAI because subtracting each plot's leaf-off
minimum is itself a per-plot constant. That is the same quantity a plot dummy absorbs, so
the two corrections overlap — see the last section.

## The ladder

Every configuration, five-fold cross-validated with folds grouped by **field campaign**.
WP0 defines a campaign as visits within three days, so two dates from one field trip are
one sampling occasion and never straddle a fold.

`calendar only` is the baseline that matters. HF069 samples March to December, so a model
that knows nothing but the date already explains most of the seasonal signal.

In [5]:
tab = (ladder[(ladder.target == "lai") & (ladder.regime == "known_plot")]
       [["features", "r2", "rmse", "mae", "n"]].reset_index(drop=True))
tab

,features,r2,rmse,mae,n
0,calendar only,0.7211,0.6790,0.5422,2894
1,evi2 only,0.8224,0.5418,0.4319,2894
2,ndvi only,0.7972,0.5789,0.4707,2894
3,spectral only,0.8452,0.5058,0.3964,2894
4,spectral + calendar,0.8620,0.4776,0.3764,2894
5,plot id + calendar,0.7714,0.6146,0.4864,2894
6,spectral + calendar + plot id,0.8977,0.4111,0.3242,2894
7,+ sza (rejected),0.8971,0.4123,0.3245,2894


In [6]:
d = tab.sort_values("r2")
base = float(tab.loc[tab.features == "calendar only", "r2"].iloc[0])
fig = go.Figure(go.Bar(
    x=d.r2, y=d.features, orientation="h",
    marker_color=["#2a78d6" if f == s8model.CHOSEN else "#a9a8a2" for f in d.features],
    text=[f"{v:.3f}" for v in d.r2], textposition="outside"))
fig.add_vline(x=base, line_dash="dash", line_color="#d95926",
              annotation_text="calendar baseline", annotation_position="top")
fig.update_layout(template="plotly_white", height=380,
                  margin=dict(l=210, r=60, t=60, b=40),
                  title="cross-validated R2, target = LAI, known plot on a new date",
                  xaxis_title="R2", xaxis_range=[0, 1], showlegend=False)
fig.show()

## Two rejected inputs

**Solar zenith angle.** At this latitude and near nadir, SZA is a function of the
calendar. Regressed on a day-of-year harmonic it leaves a residual of about a degree, and
adding it to the model changes nothing. Illumination geometry earns its keep where it
varies within a day at a fixed canopy, which is S4's design, not this one.

**MODIS.** `MCD15A2H` cells differ by roughly 0.15 LAI on a typical date, so a 500 m
product cannot resolve 10 m plots. A third of its good-quality retrievals are flagged
saturated, and its record ends 2020-12-26.

In [7]:
print(json.dumps(SUMMARY["sza_vs_calendar"], indent=2))
for f in (s8findings.f04_sza_is_the_calendar(), s8findings.f05_modis_adds_nothing()):
    print(f"\n{f.id}  {'PASS' if f.holds else 'FAIL'}\n  {f.value}")

{
  "r2": 0.9927,
  "residual_sd_deg": 1.206,
  "sza_range": [
    22.6,
    67.0
  ],
  "vza_max": 8.1
}

S8-04  PASS
  SZA on a day-of-year harmonic returns R2 0.9927, residual sd 1.21 deg, over 22.6-67.0 deg at a maximum view zenith of 8.1 deg. Three tests: as a main effect R2 0.8977 -> 0.8971; as an index interaction -> 0.8960; and the retrieval's residuals carry no SZA structure at all (r = +0.0154)

S8-05  PASS
  on the 1393 pairs MODIS reaches, adding Lai_500m and Fpar_500m moves R2 from 0.9347 to 0.9351. Its cells differ by only 0.152 LAI on a typical date and 35 % of its good-quality retrievals are flagged saturated


## Two regimes

Plot dummies answer a different question from the spectra, so the model has two honest
scores and the write-up has to name one.

- **known plot, new date** — dummies in, folds grouped by campaign. The plot appears in
  training on other dates, so its offset is learnable.
- **new plot** — dummies out, folds grouped by plot. The held-out plot is unseen.

In [8]:
pd.DataFrame([
    {"regime": "known plot, new date", **SUMMARY["known_plot"]},
    {"regime": "new plot", **SUMMARY["new_plot"]},
    {"regime": "calendar only (baseline)", **SUMMARY["baseline_calendar_only"]},
])[["regime", "r2", "rmse", "mae", "bias", "n", "n_groups"]]

,regime,r2,rmse,mae,bias,n,n_groups
0,"known plot, new date",0.8977,0.4111,0.3242,0.0004,2894,92
1,new plot,0.8629,0.4760,0.3756,0.0008,2894,37
2,calendar only (baseline),0.7211,0.6790,0.5422,0.0119,2894,92


## Where the skill lives

The pooled number hides the failure mode. Indices saturate once the canopy closes, and at
peak season the ground barely varies, so the spectral model has little to work with.
Plot identity partly covers for it — which means peak-season skill is mostly the plot's
own average rather than anything the satellite saw that week.

In [9]:
order = ["winter", "green-up", "peak", "senescence", "late"]
fig = go.Figure()
for feats, colour in [("spectral + calendar", "#2a78d6"), (s8model.CHOSEN, "#eb6834")]:
    s = seasons[seasons.features == feats].set_index("season").loc[order]
    fig.add_bar(x=order, y=s.r2, name=feats, marker_color=colour,
                text=[f"{v:.2f}" for v in s.r2], textposition="outside")
fig.update_layout(template="plotly_white", height=380, barmode="group",
                  margin=dict(l=60, r=20, t=60, b=40),
                  title="cross-validated R2 by season",
                  yaxis_title="R2", yaxis_range=[0, 1],
                  legend=dict(orientation="h", y=1.12, x=0))
fig.show()

## Residuals

Out-of-fold predictions from the shipped configuration. Bias is reported in the table
above; what matters here is whether error is flat across the range or concentrated at one
end.

In [10]:
fig = go.Figure(go.Scatter(
    x=oof.lai, y=oof.pred_lai, mode="markers",
    marker=dict(size=4, opacity=0.35, color=oof.doy, colorscale="Viridis",
                colorbar=dict(title="DOY", thickness=12)),
    hovertext=oof["plot"]))
lim = [0, float(max(oof.lai.max(), oof.pred_lai.max())) * 1.05]
fig.add_scatter(x=lim, y=lim, mode="lines", line=dict(color="#888780", dash="dash"),
                showlegend=False)
fig.update_layout(template="plotly_white", height=420,
                  margin=dict(l=60, r=20, t=60, b=40),
                  title="observed vs out-of-fold predicted LAI",
                  xaxis_title="HF069 LAI (PAI - plot minimum)",
                  yaxis_title="predicted", xaxis_range=lim, yaxis_range=lim,
                  showlegend=False)
fig.show()

In [11]:
res = (oof.assign(season=pd.cut(oof.doy, [0, 120, 160, 250, 300, 366],
                                labels=order))
       .groupby("season", observed=True)
       .agg(n=("resid", "size"), bias=("resid", "mean"),
            sd=("resid", "std"), worst=("resid", lambda s: s.abs().max())).round(3))
res

,n,bias,sd,worst
season,,,,
winter,250,0.100,0.307,1.149
green-up,534,-0.093,0.365,1.209
peak,786,0.041,0.410,2.179
senescence,765,0.001,0.464,2.007
late,559,-0.013,0.401,1.336


## PAI or LAI makes no difference to the fit

With plot dummies in the model, predicting converted LAI and predicting raw PAI give the
same RMSE. `LAI = PAI - (a per-plot constant)` is exactly what a plot dummy already
absorbs. Convert because the write-up and FT04 need leaf area, not because the model
gains anything.

In [12]:
print(json.dumps(SUMMARY["rmse_by_target"], indent=2))
ladder[(ladder.features == s8model.CHOSEN)][["target", "r2", "rmse", "target_sd"]]

{
  "lai": 0.4111,
  "pai": 0.4114
}


,target,r2,rmse,target_sd
11,lai,0.8977,0.4111,1.2856
24,pai,0.9058,0.4114,1.3405


## Forecasting a future date

The retrieval above needs the pass for the date it predicts, so it cannot reach the
future. This section asks the other question: given only what is known on `D - h`, what
is the leaf area on `D`? Satellite observations inside the horizon are withheld
entirely.

Day-of-year for the target date is legitimately known in advance, which is why the
calendar is the reference rather than an input to be credited. `persistence` runs the
retrieval on the last pass and carries that value forward — the "nothing has changed"
answer. Beating it is the real test.

In [13]:
from s8 import forecast as s8fc

fc = pd.read_csv(s8fc.OUT / "forecast_results.csv")
fc.pivot(index="horizon_days", columns="reference", values="r2")

reference,calendar,forecast,forecast + trend,persistence
horizon_days,,,,
8,0.7762,0.8172,0.8251,0.7850
16,0.7707,0.7796,0.7851,0.7475
32,0.7755,0.7746,0.7858,0.7384
64,0.7784,0.7791,0.7793,0.7272


In [14]:
piv = fc.pivot(index="horizon_days", columns="reference", values="r2")
fig = go.Figure()
for name, colour, dash in [("calendar", "#a9a8a2", "dash"),
                           ("persistence", "#eb6834", "dot"),
                           ("forecast + trend", "#2a78d6", "solid")]:
    fig.add_scatter(x=piv.index, y=piv[name], mode="lines+markers", name=name,
                    line=dict(color=colour, dash=dash, width=2),
                    marker=dict(size=8))
fig.update_layout(template="plotly_white", height=380,
                  margin=dict(l=60, r=20, t=60, b=40),
                  title="forecast skill against horizon",
                  xaxis_title="horizon (days ahead)", yaxis_title="R2",
                  yaxis_range=[0.7, 0.85],
                  legend=dict(orientation="h", y=1.12, x=0))
fig.show()

Spectral memory fades; seasonal climatology does not. The calendar line is flat because
day-of-year is as informative 64 days out as it is 8. Carrying the last retrieval forward
decays steadily. The satellite's advantage is real at 8 days and gone by about a month.

Reaching further than that needs growing degree days, chilling and photoperiod — a
climate-driven phenology model, not more spectral features.

## Carried forward

- **Peak-season retrieval rests on the plot offset, not the spectra.** Any claim about
  peak leaf area needs a different instrument or an explicit caveat.
- **Clumping is uncorrected.** Optical methods read low because foliage clusters; true
  LAI is roughly the optical value divided by a clumping index near 0.9 for this stand.
  The subtraction also over-corrects, since summer leaves hide branches that were fully
  exposed at leaf-off.
- **`hf069-05-litter.csv` is the independent check** and has not been used. Litter traps
  give leaf mass on the same plots; converting to area needs specific leaf area per
  species, which is not in the file.
- **HF069 data-use permission is unanswered** since Aug 2026, and the new lead arrives
  November 2026.